# СимургФарм — итоговые физиологические модели

Версия параметрической модели из `physiology_model.py`. Загрузите `expamlomycin_train.csv` и `expamlomycin_test.csv` в рабочую папку Colab, затем выполните все ячейки. Прогнозы предназначены исключительно для конкурсного кейса.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT/'expamlomycin_train.csv').exists() and (ROOT/'expamlomycin_test.csv').exists(), 'Загрузите оба исходных CSV в Colab'
train = pd.read_csv(ROOT/'expamlomycin_train.csv')
test = pd.read_csv(ROOT/'expamlomycin_test.csv', sep=';')

## 1. Признаки, обучение CL и Vd
Степенная регрессия на основе Кокрофта — Голта, OAT, массы, сепсиса и альбумина. Резервные регрессии для отсутствующих анализов. Коэффициенты оптимизируются по MAPE.

In [ ]:
def features(d):
    a=d.age.to_numpy(float); w=d.weight_kg.to_numpy(float); s=d.sepsis_status.to_numpy(float)
    f=(d.sex=='F').to_numpy(float); slow=(d.oat_variant=='Slow').to_numpy(float); rapid=(d.oat_variant=='Rapid').to_numpy(float)
    scr=d.scr_umol_l.to_numpy(float); alb=d.albumin_g_l.to_numpy(float)
    cr=(140-a)*w/(.814*scr)*np.where(f==1,.85,1.)
    return a,w,s,f,slow,rapid,scr,alb,cr

def design(d):
    a,w,s,f,slow,rapid,scr,alb,cr=features(d)
    one=np.ones(len(d)); lw=np.log(w/70); la=np.log((140-a)/80)
    Xcl=np.column_stack([one,np.log(cr/100),slow,rapid])
    # Reserve model uses demographic predictors and clinical state, without creatinine.
    Xfallback=np.column_stack([one,lw,la,f,slow,rapid,s])
    Xvd=np.column_stack([one,lw,s,(40-alb)/10])
    Xvd_simple=np.column_stack([one,lw,s])
    return Xcl,Xfallback,Xvd,Xvd_simple,np.isfinite(scr),np.isfinite(alb)

def fit_log(X,y):
    y=np.asarray(y); b=np.linalg.lstsq(X,np.log(y),rcond=None)[0]
    # Fit coefficients to the actual contest MAPE objective.
    def loss(b): return np.mean(np.abs(np.exp(X@b)/y-1))
    opt=minimize(loss,b,method='Powell',options={'maxiter':500,'xtol':1e-8,'ftol':1e-10})
    return opt.x if np.isfinite(opt.fun) and opt.fun<=loss(b) else b

def fit(d):
    cl,fb,vd,vs,sc,al=design(d)
    ycl=d.CL_L_h.to_numpy(); yvd=d.Vd_L.to_numpy()
    return [fit_log(cl[sc],ycl[sc]),fit_log(fb,ycl),fit_log(vd[al],yvd[al]),fit_log(vs,yvd)]

def predict(d,b):
    cl,fb,vd,vs,sc,al=design(d)
    pcl=np.exp(fb@b[1]); pcl[sc]=np.exp(cl[sc]@b[0]);
    pvd=np.exp(vs@b[3]); pvd[al]=np.exp(vd[al]@b[2]);
    return np.column_stack([pcl,pvd])

def oof(d,seed=42):
    p=np.empty((len(d),2));
    for ii,jj in KFold(5,shuffle=True,random_state=seed).split(d):
        p[jj]=predict(d.iloc[jj],fit(d.iloc[ii]))
    return p

## 2. Расчёт дозы и метрика
Допустимые границы дозы и выбор дозы по эмпирическим OOF-ошибкам.

In [ ]:
def bounds(cl,vd):
    k=cl/vd
    a= -np.expm1(-12*k)
    lo=72*cl/a
    hi=np.minimum(144*cl/a,26*vd*np.exp(.5*k))
    return lo,hi

def dose(p, d, cal_p, cal_d):
    # Estimate empirical prediction error distribution from training-only OOF predictions.
    r=cal_d[['CL_L_h','Vd_L']].to_numpy()/cal_p
    cm=cal_d.scr_umol_l.isna().to_numpy(); am=cal_d.albumin_g_l.isna().to_numpy(); cs=cal_d.sepsis_status.to_numpy()
    out=[]
    for (cl,vd),(_,x) in zip(p,d.iterrows()):
        lo,hi=bounds(cl,vd)
        if lo>hi:
            out.append(hi*(1-1e-9));continue
        miss=pd.isna(x.scr_umol_l); amiss=pd.isna(x.albumin_g_l); s=x.sepsis_status
        if miss:
            # Simulated missing-SCr predictions on all calibration subjects are available via direct fallback separately only in training; use actual group and pool over sepsis if small.
            mask=cm
        else:
            mask=(~cm)&(cs==s)&(am==amiss)
            if mask.sum()<80: mask=(~cm)&(cs==s)
        if mask.sum()<20: mask=np.ones(len(r),bool)
        rr=r[mask]; low,high=bounds(cl*rr[:,0],vd*rr[:,1]); valid=low<=high
        low,high=low[valid],high[valid]
        candidates=np.linspace(lo,hi,301)
        # Smooth empirical coverage slightly to reduce unstable choices in small groups.
        # Maximize score over a narrow dose uncertainty band and prefer interior candidates on ties.
        hit=((candidates[:,None]>=low)&(candidates[:,None]<=high)).mean(axis=1) if len(low) else np.zeros(len(candidates))
        smooth=np.convolve(np.pad(hit,(5,5),mode='edge'),np.ones(11)/11,mode='valid')
        j=np.argmax(smooth-1e-5*np.abs(np.arange(len(candidates))-150))
        out.append(float(np.clip(candidates[j],lo+1e-8*(hi-lo),hi-1e-8*(hi-lo))))
    return np.array(out)

def score(d,p,doses):
    y=d[['CL_L_h','Vd_L']].to_numpy(); m=np.mean(abs(p/y-1),axis=0)*100
    lo,hi=bounds(y[:,0],y[:,1]); tar=100*np.mean((doses>=lo)&(doses<=hi));
    return m,tar,(100-m.mean()+tar)/2

## 3. Вложенная кросс-валидация и обучение на всех данных

In [ ]:
def main():
    p=np.empty((len(train),2)); doses=np.empty(len(train)); simple=np.empty(len(train))
    # Outer validation: dose calibration only on inner OOF predictions, never on outer validation labels.
    for fold,(ii,jj) in enumerate(KFold(5,shuffle=True,random_state=42).split(train)):
        tr=train.iloc[ii]; va=train.iloc[jj]; p[jj]=predict(va,fit(tr));
        cp=oof(tr,seed=123); doses[jj]=dose(p[jj],va,cp,tr)
        lo,hi=bounds(p[jj,0],p[jj,1]); simple[jj]=np.where(lo<=hi,np.sqrt(lo*hi),hi)
        print('Fold',fold+1, score(va,p[jj],doses[jj]),flush=True)
    print('OOF calibrated',score(train,p,doses));print('OOF midpoint',score(train,p,simple))
    b=fit(train);print('Coefficients',b)
    pred=predict(test,b); ds=dose(pred,test,p,train)
    sub=pd.DataFrame({'patient_id':test.patient_id,'Vd_L':pred[:,1],'CL_L_h':pred[:,0],'Dose_mg':ds})
    # 10 significant decimal places preserve constraints except for impossible cases.
    sub.to_csv(ROOT/'submission_physiology.csv',index=False,float_format='%.10f')
    read=pd.read_csv(ROOT/'submission_physiology.csv');assert len(read)==500 and read.patient_id.equals(test.patient_id) and read.patient_id.is_unique and not read.isna().any().any() and (read.iloc[:,1:]>0).all().all()
    lo,hi=bounds(read.CL_L_h.to_numpy(),read.Vd_L.to_numpy());print('test feasibility',sum(lo<=hi),'dose within limits',sum((read.Dose_mg>=lo-1e-7)&(read.Dose_mg<=hi+1e-7)))
    print(read.head().to_string(index=False))
    for status in [False,True]:
        sel=train.scr_umol_l.isna()==status; print('CL missing',status,np.mean(abs(p[sel,0]/train.CL_L_h[sel]-1))*100)

main()

## 4. CSV в официальном формате
Переименовываем и округляем столбцы в соответствии с шаблоном организаторов.

In [ ]:
raw = pd.read_csv(ROOT/'submission_physiology.csv')
submission = pd.DataFrame({
    'patient_id': raw['patient_id'],
    'pred_CL': raw['CL_L_h'].round(3),
    'pred_Vd': raw['Vd_L'].round(2),
    'Dose': raw['Dose_mg']
})
assert len(submission) == len(test)
assert submission.patient_id.equals(test.patient_id)
assert not submission.isna().any().any()
submission.to_csv(ROOT/'submission_physiology_corrected.csv', index=False)
submission.head()